In [1]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
import scipy.sparse as sp
import yaml
import time
import gget

import anndata as an
import scanpy as sc
import rapids_singlecell as rsc
import scvi

sc.settings.verbosity = 3
sc.logging.print_header()

10:08:45 - INFO - Old pandas version detected. Patching DataFrame.map to DataFrame.applymap
/nfs/turbo/umms-indikar/Cooper/conda_envs/scrapids/lib/python3.12/site-packages/docrep/decorators.py:43: SyntaxWarning: 'param_categorical_covariate_keys' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)
/nfs/turbo/umms-indikar/Cooper/conda_envs/scrapids/lib/python3.12/site-packages/docrep/decorators.py:43: SyntaxWarning: 'param_continuous_covariate_keys' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)


Package,Version
ipykernel,6.29.5
numpy,1.26.4
pandas,2.0.3
torch,2.5.1.post207
ipywidgets,8.1.5
xarray,2025.3.0
matplotlib,3.10.1
seaborn,0.13.2
h5py,3.13.0
scipy,1.15.2


In [2]:
fpath = "/scratch/indikar_root/indikar1/shared_data/HYB/anndata/raw_epi2me_merged.h5ad"
adata = sc.read_h5ad(fpath)
sc.logging.print_memory_usage()
adata

Memory usage: current 1.48 GB, difference +1.48 GB


AnnData object with n_obs × n_vars = 19760 × 25549
    obs: 'MYOD_counts', 'PRRX1_counts', 'PRRX1_MYOD_counts', 'prediction', 'total_fb_counts', 'rate_prediction', 'G1_counts', 'G2M_counts', 'S_counts', 'dataset', 'total_reads', 'total_genes'

In [3]:
adata.obs.head()

,MYOD_counts,PRRX1_counts,PRRX1_MYOD_counts,prediction,total_fb_counts,rate_prediction,G1_counts,G2M_counts,S_counts,dataset,total_reads,total_genes
cell_barcode,,,,,,,,,,,,
AAACCAAAGCAACTGC,0.0,0.0,25.0,PRRX1_MYOD,25.0,1.000000,NaN,NaN,NaN,Hybrid,12782.0,3789
AAACCAAAGCTATGAT,1.0,1.0,6.0,PRRX1_MYOD,8.0,0.750000,NaN,NaN,NaN,Hybrid,1085.0,636
AAACCAAAGTAGCCGT,1.0,0.0,21.0,PRRX1_MYOD,22.0,0.954545,NaN,NaN,NaN,Hybrid,1306.0,736
AAACCAAAGTAGGGCA,14.0,1.0,1.0,MYOD,16.0,0.875000,NaN,NaN,NaN,Hybrid,6666.0,2750
AAACCAAAGTCTAGGC,0.0,0.0,25.0,PRRX1_MYOD,25.0,1.000000,NaN,NaN,NaN,Hybrid,19285.0,5030


In [4]:
adata.raw = adata  # keep full dimension safe

adata.layers['raw_counts'] = adata.X.copy()

rsc.get.anndata_to_GPU(adata)
rsc.pp.filter_cells(adata, min_counts=200)
rsc.pp.filter_genes(adata, min_counts=50)
rsc.pp.normalize_total(adata, target_sum=1e4)
rsc.pp.log1p(adata)

adata.layers['log_norm'] = adata.X.get()

rsc.get.anndata_to_GPU(adata) # move to GPU
adata

CUDARuntimeError: cudaErrorInsufficientDriver: CUDA driver version is insufficient for CUDA runtime version

In [ ]:
print(f"Number of genes before HVG selection: {adata.n_vars}")

rsc.pp.highly_variable_genes(
    adata, 
    flavor="seurat_v3", 
)

print(f"Number of HVG: {adata.var['highly_variable'].sum()}")

plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 5, 4
sc.pl.highly_variable_genes(
    adata
)

adata

In [ ]:
rsc.pp.pca(
    adata,
)

plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 5, 3
sc.pl.pca_variance_ratio(
    adata
)

adata

In [ ]:
rsc.pp.neighbors(
    adata,
)

rsc.tl.umap(
    adata,
)

rsc.tl.leiden(
    adata,
    resolution=0.35,
)

plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 6, 6

sc.pl.umap(
    adata,
    ncols=1,
    color=['dataset', 'leiden', 'prediction']
)


# SCVI

In [ ]:
scvi.model.SCVI.setup_anndata(
    adata, 
    layer="raw_counts",
    batch_key="dataset",
)

model = scvi.model.SCVI(
    adata,
    n_layers=2,
    n_latent=24,
    gene_likelihood="nb",
)

model.train(
    max_epochs=100,
    accelerator="gpu",
    devices="auto",
    enable_model_summary=True,
    batch_size=1000,
    early_stopping=True,
    early_stopping_patience=2,
    early_stopping_monitor='validation_loss',
)

adata.obsm["X_scVI"] = model.get_latent_representation()
print(f"{adata.obsm["X_scVI"].shape=}")

In [ ]:
rsc.pp.neighbors(adata, use_rep="X_scVI", n_neighbors=15)
rsc.tl.leiden(adata, resolution=0.2)

rsc.tl.umap(adata, min_dist=0.25)

sc.pl.umap(
    adata,
    color=['dataset', 'leiden', 'prediction'],
    frameon=False,
    ncols=1,
)

# Get normalized expression values

In [ ]:
adata.layers['scvi_counts'] = model.get_normalized_expression(
    return_mean=False,
)

sc.pl.umap(
    adata,
    layer='scvi_counts',
    color=['prediction', 'MYOD1', 'PRRX1'],
    ncols=2,
    size=10,
)

adata

In [ ]:
sc.pl.umap(
    adata,
    layer='log_norm',
    color=['prediction', 'MYOD1', 'PRRX1'],
    ncols=2,
    size=10,
)

# Differential Expression with scVI

In [ ]:
deg = model.differential_expression(
    adata,
    groupby='prediction',
    batch_correction=True,
    filter_outlier_cells=True,
)

deg = deg.reset_index()
print(f"{deg.shape=}")
deg.head()

In [ ]:
deg.columns

In [ ]:
n_genes_per_group = 100
# database = 'celltypes'
database = 'ontology'

for group_name, group_df in deg.groupby('group1'):
    group_df = group_df.sort_values(by='lfc_mean', ascending=False)

    group_df = group_df.head(n_genes_per_group)
    
    gene_list = group_df['gene_name'].to_list()
    edf = gget.enrichr(gene_list, database=database)

    print(f"\n ------ {group_name} ------ ")
    print(edf[['path_name', 'adj_p_val', 'overlapping_genes']].head())
    

    # break